# Phase 2: Baseline Data Preparation and EDA

This notebook converts the Phase 1 decisions into a reproducible household-level dataset. The baseline population is defined by the 16,057 households with a valid agricultural-input record in `S5_Agricultural Inputs.dta`. `idquest` is the merge key. The baseline feature sources are `S0` general information, `S2` land and crops, `S6` agricultural practices, and aggregated `S9` livestock data.


## Preparation principles

- Keep exactly one row per household after every merge.
- Use `s5q1_1` only to construct the target; do not use the remaining input-response fields as baseline predictors because they can directly reveal the target.
- Aggregate repeated `S9` records before merging. Numeric livestock quantities are summed and categorical descriptors use the modal non-missing response.
- Preserve missing values during the audit. Imputation and encoding will be fitted inside the modeling pipeline in the next phase to avoid leakage from the test set.
- Save both the prepared table and an audit report so the final sample and feature provenance are reproducible.

In [10]:
import pandas as pd
from pathlib import Path

base_path = Path('stata')
base = pd.read_stata(base_path / 'S0_General Information.dta')
land = pd.read_stata(base_path / 'S2_Land tenure and crops planted.dta')
inputs = pd.read_stata(base_path / 'S5_Agricultural Inputs.dta')
practices = pd.read_stata(base_path / 'S6_Agricultural Practices.dta')
animals = pd.read_stata(base_path / 'S9_Number of animals.dta')

print('base shape:', base.shape)
print('land shape:', land.shape)
print('inputs shape:', inputs.shape)
print('practices shape:', practices.shape)
print('animals shape:', animals.shape)
print('household id present in base?', 'idquest' in base.columns)

base shape: (23709, 25)
land shape: (16057, 30)
inputs shape: (16057, 162)
practices shape: (16057, 40)
animals shape: (32611, 38)
household id present in base? True


In [5]:
def mode_non_null(series):
    values = series.dropna()
    return values.mode().iloc[0] if not values.empty else pd.NA

def aggregate_household_records(frame):
    aggregations = {}
    for column in frame.columns:
        if column == 'idquest':
            continue
        if pd.api.types.is_numeric_dtype(frame[column]):
            aggregations[column] = 'sum'
        else:
            aggregations[column] = mode_non_null
    return frame.groupby('idquest', as_index=False, sort=False).agg(aggregations)

# S0 contains a small number of invalid and duplicate records. Use the clean
# target population as the spine, then retain one valid S0 row per household.
inputs = inputs.copy()
inputs['target'] = inputs['s5q1_1'].map({'Yes': 1, 'No': 0}).astype('Int64')
target_table = inputs[['idquest', 'target']].copy()
target_table = target_table.dropna(subset=['idquest', 'target']).drop_duplicates('idquest')

base_clean = (
    base.loc[base['idquest'].notna(), ['idquest', 's0q1', 's0q2', 's0q3', 's0q4']]
    .drop_duplicates('idquest')
)
land_core = land.drop_duplicates('idquest')
practices_core = practices.drop_duplicates('idquest')
animal_columns = ['idquest'] + [column for column in animals.columns if column.startswith('s9')]
animals_agg = aggregate_household_records(animals[animal_columns])

# Select non-target features from the core modules.
land_columns = ['idquest', 's2q1', 's2q2', 's2q3', 's2q4', 's2q5', 's2q6', 's2q7_1', 's2q7_2', 's2q7_3', 's2q15_1', 's2q15_2', 's2q15_3']
practice_columns = ['idquest', 'dummy_cropping_B', 'cropping', 's6q1_1', 's6q1_2_1', 's6q1_2_2', 's6q1_2_3', 's6q1_3', 's6q1_4_1', 's6q1_4_2', 's6q1_4_3', 's6q2_1', 's6q2_2_1']
land_core = land_core[[column for column in land_columns if column in land_core.columns]]
practices_core = practices_core[[column for column in practice_columns if column in practices_core.columns]]

prepared = target_table.merge(base_clean, on='idquest', how='left', validate='one_to_one')
prepared = prepared.merge(land_core, on='idquest', how='left', validate='one_to_one')
prepared = prepared.merge(practices_core, on='idquest', how='left', validate='one_to_one', suffixes=('', '_practice'))
prepared = prepared.merge(animals_agg, on='idquest', how='left', validate='one_to_one', suffixes=('', '_animal'))

assert prepared['idquest'].notna().all()
assert prepared['idquest'].is_unique
assert prepared['target'].isin([0, 1]).all()
print('Prepared shape:', prepared.shape)
print('Unique households:', prepared['idquest'].nunique())
print('Target distribution:', prepared['target'].value_counts().sort_index().to_dict())
print('Duplicate household IDs:', prepared['idquest'].duplicated().sum())

Prepared shape: (16057, 62)
Unique households: 16057
Target distribution: {np.int64(0): 4977, np.int64(1): 11080}
Duplicate household IDs: 0


## Target definition and leakage control

The target is `target = 1` when `s5q1_1 == 'Yes'` and `target = 0` when `s5q1_1 == 'No'`. The other `S5` input-response fields are deliberately excluded from the baseline feature table. Although they may be predictive, they describe the same input-use decision and could make the classifier appear accurate by revealing the answer rather than learning from household characteristics.


The baseline predictors therefore come from household context, land and crop characteristics, agricultural practices, and aggregated livestock characteristics. Optional modules will be tested only after this baseline has been evaluated.

In [6]:
# The target was created during the merge using an explicit Yes/No mapping.
# Confirm the raw values were fully accounted for before saving the table.
raw_target_values = inputs['s5q1_1'].value_counts(dropna=False)
assert set(raw_target_values.index) <= {'Yes', 'No'}
assert inputs['target'].notna().all()

target_summary = pd.DataFrame({
    'class': ['No', 'Yes'],
    'encoded_value': [0, 1],
    'households': [int((prepared['target'] == 0).sum()), int((prepared['target'] == 1).sum())],
})
target_summary['share'] = (target_summary['households'] / len(prepared)).round(4)
target_summary

,class,encoded_value,households,share
0,No,0,4977,0.31
1,Yes,1,11080,0.69


## Missingness, feature audit, and export

Missing values are reported before any imputation. This is important because the extent and pattern of missingness determine whether a variable is suitable for modeling. We do not impute here; the next modeling phase will fit imputation and encoding steps using training data only.


The feature audit records each column's source group, data type, missing count, and number of distinct values. This makes the prepared dataset explainable and provides a direct input to the Methods section of the report.

In [7]:
source_groups = {
    'idquest': 'identifier',
    'target': 'target',
    's0': 'household_context',
    's2': 'land_and_crops',
    's6': 'agricultural_practices',
    's9': 'livestock_aggregated',
}

def source_group(column):
    if column in source_groups:
        return source_groups[column]
    for prefix, group in [('s0', 'household_context'), ('s2', 'land_and_crops'), ('s6', 'agricultural_practices'), ('s9', 'livestock_aggregated')]:
        if column.startswith(prefix):
            return group
    if column.endswith('_practice'):
        return 'agricultural_practices'
    if column.endswith('_animal') or '_animal' in column:
        return 'livestock_aggregated'
    return 'review_required'

feature_audit = pd.DataFrame({
    'column': prepared.columns,
    'source_group': [source_group(column) for column in prepared.columns],
    'dtype': prepared.dtypes.astype(str).values,
    'missing_count': prepared.isna().sum().values,
    'missing_share': prepared.isna().mean().round(4).values,
    'distinct_values': prepared.nunique(dropna=True).values,
})
display(feature_audit)

missingness_summary = (
    feature_audit.sort_values(['missing_share', 'column'], ascending=[False, True])
    .reset_index(drop=True)
 )
print('Columns with missing values:', int((feature_audit['missing_count'] > 0).sum()))
display(missingness_summary.head(20))

output_path = Path('prepared_household_data.csv')
audit_path = Path('phase_2_feature_audit.csv')
prepared.to_csv(output_path, index=False)
feature_audit.to_csv(audit_path, index=False)
print(f'Saved {output_path} with shape {prepared.shape}')
print(f'Saved {audit_path} with {len(feature_audit)} feature records')

,column,source_group,dtype,missing_count,missing_share,distinct_values
0,idquest,identifier,float64,0,0.0,16057
1,target,target,Int64,0,0.0,2
2,s0q1,household_context,category,0,0.0,5
3,s0q2,household_context,category,0,0.0,30
4,s0q3,household_context,category,0,0.0,5
...,...,...,...,...,...,...
57,s9q9_24,livestock_aggregated,float64,0,0.0,39
58,s9q9_25,livestock_aggregated,float64,0,0.0,70
59,s9q9_26,livestock_aggregated,float64,0,0.0,31
60,s9q9_27,livestock_aggregated,float64,0,0.0,22


Columns with missing values: 16


,column,source_group,dtype,missing_count,missing_share,distinct_values
0,s6q1_4_3,agricultural_practices,category,15563,0.9692,4
1,s6q1_2_3,agricultural_practices,category,15030,0.9360,9
2,s6q2_2_1,agricultural_practices,category,14562,0.9069,5
3,s6q1_4_2,agricultural_practices,category,13853,0.8627,4
4,s6q1_2_2,agricultural_practices,category,10585,0.6592,9
5,s2q15_3,land_and_crops,category,8843,0.5507,53
6,s6q1_4_1,agricultural_practices,category,7945,0.4948,4
7,s2q7_3,land_and_crops,category,7702,0.4797,48
8,s6q1_2_1,agricultural_practices,category,6220,0.3874,9
9,s9q9_1,livestock_aggregated,category,4412,0.2748,15


Saved prepared_household_data.csv with shape (16057, 62)
Saved phase_2_feature_audit.csv with 62 feature records


## Initial exploratory data analysis

This first EDA pass describes the modeling population before any model-specific preprocessing. It checks the target balance, numerical distributions, and the most incomplete variables. These outputs provide the baseline facts that will guide imputation, encoding, and feature selection in Phase 3.

In [8]:
numeric_features = prepared.select_dtypes(include='number').drop(columns=['idquest'], errors='ignore')
numeric_summary = numeric_features.describe().T
numeric_summary['missing_count'] = numeric_features.isna().sum()
numeric_summary['missing_share'] = numeric_features.isna().mean().round(4)
display(numeric_summary)

categorical_features = prepared.select_dtypes(include=['object', 'category'])
categorical_summary = pd.DataFrame({
    'column': categorical_features.columns,
    'distinct_values': categorical_features.nunique(dropna=True).values,
    'missing_count': categorical_features.isna().sum().values,
    'missing_share': categorical_features.isna().mean().round(4).values,
})
display(categorical_summary.sort_values('missing_share', ascending=False))

print('Phase 2 summary')
print(f'Households: {len(prepared):,}')
print(f'Features excluding idquest and target: {prepared.shape[1] - 2}')
print(f'Numerical features: {len(numeric_features.columns)}')
print(f'Categorical features: {len(categorical_features.columns)}')
print(f'Target prevalence: {prepared["target"].mean():.3f}')

,count,mean,std,min,25%,50%,75%,max,missing_count,missing_share
target,16057.0,0.690042,0.462491,0.0,0.0,1.0,1.0,1.0,0,0.0
s0q4,16057.0,17.407237,13.777007,1.0,7.0,15.0,23.0,72.0,0,0.0
dummy_cropping_B,16057.0,1.05823,0.234185,1.0,1.0,1.0,1.0,2.0,0,0.0
cropping,16057.0,1.035498,0.185042,1.0,1.0,1.0,1.0,2.0,0,0.0
s9q9_3,16057.0,4.854892,39.228233,0.0,0.0,2.0,5.0,4038.0,0,0.0
s9q9_4_1,16057.0,0.111665,0.630897,0.0,0.0,0.0,0.0,30.0,0,0.0
s9q9_4_2,16057.0,0.439185,1.355911,0.0,0.0,0.0,1.0,90.0,0,0.0
s9q9_5_1,16057.0,1.003114,2.20592,0.0,0.0,0.0,1.0,70.0,0,0.0
s9q9_5_2,16057.0,0.206078,0.863904,0.0,0.0,0.0,0.0,20.0,0,0.0
s9q9_5_3,16057.0,0.691287,1.57709,0.0,0.0,0.0,1.0,39.0,0,0.0


,column,distinct_values,missing_count,missing_share
22,s6q1_4_3,4,15563,0.9692
18,s6q1_2_3,9,15030,0.9360
24,s6q2_2_1,5,14562,0.9069
21,s6q1_4_2,4,13853,0.8627
17,s6q1_2_2,9,10585,0.6592
14,s2q15_3,53,8843,0.5507
20,s6q1_4_1,4,7945,0.4948
11,s2q7_3,48,7702,0.4797
16,s6q1_2_1,9,6220,0.3874
26,s9q9_2,1,4412,0.2748


Phase 2 summary
Households: 16,057
Features excluding idquest and target: 60
Numerical features: 34
Categorical features: 27
Target prevalence: 0.690


## Phase 2 conclusion and handoff

The baseline preparation is accepted for modeling. It contains 16,057 unique households and preserves the verified class distribution: 4,977 non-users (31%) and 11,080 users (69%). The prepared table contains 60 candidate predictors before model-specific filtering: 34 numerical and 27 categorical variables, plus `idquest` and `target`.

The largest missingness levels occur in optional agricultural-practice and crop-detail fields. These fields are retained for now because missingness may represent a meaningful survey skip, but the modeling pipeline must impute and encode them using training data only. The identifier will be removed before modeling. The next phase can therefore begin with a leakage-safe train/test split and comparable classification models.